# 13 — MURA all-anatomy development evaluation

This notebook evaluates the all-anatomy baseline on the internal validation set. It is not an official-test result and must not be described as clinical performance.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

def locate_project_root():
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "CHARTER.md").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the repository checkout.")

ROOT = locate_project_root()
RUNTIME = ROOT.parent / 'runtime' / 'multianatomy_v1'
MANIFESTS = RUNTIME / 'manifests'
QC = RUNTIME / 'qc'
BASELINE = RUNTIME / 'baseline_all_v1'
assert ROOT.name == 'mura-wrist-ml-pipeline', ROOT
print('Repository: local project checkout')
print('Runtime artifacts: local only (not committed to Git)')


Repository: local project checkout
Runtime artifacts: local only (not committed to Git)


In [2]:
metrics = pd.read_csv(BASELINE / 'baseline_metrics_by_anatomy.csv')
thresholds = pd.read_csv(BASELINE / 'baseline_thresholds_by_anatomy.csv')
bootstrap = pd.read_csv(BASELINE / 'baseline_patient_bootstrap_ci.csv')
assert metrics['official_valid_read'].eq(False).all()
assert thresholds['official_valid_read'].eq(False).all()
display(Markdown('**Evaluation scope:** internal validation only; no official MURA `valid` label was read.'))
display(thresholds.round(4))

**Evaluation scope:** internal validation only; no official MURA `valid` label was read.

,anatomy,threshold,target_sensitivity,achieved_sensitivity,achieved_specificity,candidates_evaluated,selection_rule,selection_population,official_valid_read
0,XR_ELBOW,0.0409,0.95,0.9538,0.2651,345,highest_threshold_with_sensitivity_at_least_ta...,internal_validation,False
1,XR_FINGER,0.0215,0.95,0.9535,0.2053,392,highest_threshold_with_sensitivity_at_least_ta...,internal_validation,False
2,XR_FOREARM,0.0493,0.95,0.9636,0.3672,183,highest_threshold_with_sensitivity_at_least_ta...,internal_validation,False
3,XR_HAND,0.0151,0.95,0.9533,0.0724,411,highest_threshold_with_sensitivity_at_least_ta...,internal_validation,False
4,XR_HUMERUS,0.0670,0.95,0.9636,0.4921,118,highest_threshold_with_sensitivity_at_least_ta...,internal_validation,False
5,XR_SHOULDER,0.1184,0.95,0.9522,0.2920,567,highest_threshold_with_sensitivity_at_least_ta...,internal_validation,False
6,XR_WRIST,0.0530,0.95,0.9522,0.3624,708,highest_threshold_with_sensitivity_at_least_ta...,internal_validation,False


In [3]:
summary = metrics[[
    'anatomy', 'auroc', 'auprc', 'sensitivity', 'specificity', 'precision',
    'f1', 'false_negative_rate', 'false_negative', 'false_positive', 'flagged_percent'
]].copy()
display(summary.round(4))

,anatomy,auroc,auprc,sensitivity,specificity,precision,f1,false_negative_rate,false_negative,false_positive,flagged_percent
0,XR_ELBOW,0.8550,0.8367,0.9538,0.2651,0.4397,0.6019,0.0462,6,158,81.7391
1,XR_FINGER,0.7465,0.6010,0.9535,0.2053,0.3705,0.5336,0.0465,6,209,84.6939
2,XR_FOREARM,0.8439,0.6861,0.9636,0.3672,0.3955,0.5608,0.0364,2,81,73.2240
3,XR_HAND,0.7123,0.5833,0.9533,0.0724,0.2656,0.4155,0.0467,5,282,93.4307
4,XR_HUMERUS,0.7948,0.7243,0.9636,0.4921,0.6235,0.7571,0.0364,2,32,72.0339
5,XR_SHOULDER,0.8124,0.8364,0.9522,0.2920,0.5899,0.7285,0.0478,14,194,83.4215
6,XR_WRIST,0.8655,0.8372,0.9522,0.3624,0.4823,0.6403,0.0478,13,278,75.8475


In [4]:
ci_table = bootstrap.pivot(index='anatomy', columns='metric', values=['estimate', 'lower_95_ci', 'upper_95_ci'])
display(ci_table.round(4))

estimate                                                    \
metric         auprc   auroc      f1 precision sensitivity specificity   
anatomy                                                                  
XR_ELBOW      0.8367  0.8550  0.6019    0.4397      0.9538      0.2651   
XR_FINGER     0.6010  0.7465  0.5336    0.3705      0.9535      0.2053   
XR_FOREARM    0.6861  0.8439  0.5608    0.3955      0.9636      0.3672   
XR_HAND       0.5833  0.7123  0.4155    0.2656      0.9533      0.0724   
XR_HUMERUS    0.7243  0.7948  0.7571    0.6235      0.9636      0.4921   
XR_SHOULDER   0.8364  0.8124  0.7285    0.5899      0.9522      0.2920   
XR_WRIST      0.8372  0.8655  0.6403    0.4823      0.9522      0.3624   

            lower_95_ci                                                    \
metric            auprc   auroc      f1 precision sensitivity specificity   
anatomy                                                                     
XR_ELBOW         0.7858  0.8138  0.5454    0.3833      0.9173      0.2085   
XR_FINGER        0.5089  0.6920  0.4764    0.3167      0.9127      0.1560   
XR_FOREARM       0.5487  0.7762  0.4666    0.3083      0.9091      0.2880   
XR_HAND          0.4873  0.6437  0.3570    0.2205      0.9083      0.0455   
XR_HUMERUS       0.5936  0.7095  0.6667    0.5125      0.9057      0.3676   
XR_SHOULDER      0.7938  0.7758  0.6931    0.5472      0.9262      0.2403   
XR_WRIST         0.7990  0.8374  0.6022    0.4420      0.9250      0.3188   

            upper_95_ci                                                    
metric            auprc   auroc      f1 precision sensitivity specificity  
anatomy                                                                    
XR_ELBOW         0.8849  0.8958  0.6560    0.4982      0.9852      0.3255  
XR_FINGER        0.6918  0.7947  0.5864    0.4241      0.9846      0.2548  
XR_FOREARM       0.8133  0.9052  0.6392    0.4779      1.0000      0.4511  
XR_HAND          0.6766  0.7770  0.4679    0.3103      0.9905      0.1031  
XR_HUMERUS       0.8551  0.8669  0.8302    0.7250      1.0000      0.6167  
XR_SHOULDER      0.8756  0.8479  0.7635    0.6346      0.9756      0.3467  
XR_WRIST         0.8731  0.8934  0.6778    0.5259      0.9752      0.4082

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
plot = metrics.set_index('anatomy')[['auroc', 'auprc']].sort_index()
plot.plot(kind='bar', ax=axes[0], color=['#2563eb', '#0f766e'])
axes[0].set(title='Ranking metrics by anatomy', xlabel='Anatomy', ylabel='Score', ylim=(0, 1.05))
axes[0].grid(axis='y', alpha=0.25)
metrics.sort_values('anatomy').plot.bar(x='anatomy', y=['false_negative', 'false_positive'], ax=axes[1], color=['#b91c1c', '#d97706'])
axes[1].set(title='Validation errors at the fixed 95% sensitivity policy', xlabel='Anatomy', ylabel='Studies')
axes[1].grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

C:\Users\netaf\AppData\Local\Temp\ipykernel_41136\3344366813.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interpretation boundary

Differences between body parts can reflect sample size, prevalence, acquisition differences, and task difficulty. These results establish a reproducible development baseline only. A future fine-tuned model must be compared against this baseline on the same internal validation studies, with its own validation-selected threshold.